# VigiLLM teacher-query pipeline (Kaggle GPU)

Runs `scripts/query_teachers.py --backend hf` on Kaggle's free T4/P100 GPU instead of pulling OpenBioLLM-8B / BioMistral-7B via Ollama on local hardware.

**Before running:** Settings (right sidebar) -> Accelerator -> GPU T4 x2 (or P100). Internet must be ON (needed for `git clone` and HF Hub downloads).

**Repo state required:** `scripts/query_teachers.py`, `scripts/rule_engine/`, and the `--backend hf` support must be committed and pushed to `origin/main` first -- this notebook clones from GitHub, it doesn't see your local uncommitted files.

In [ ]:
# If the repo is private, add a GitHub PAT as a Kaggle secret named GITHUB_TOKEN
# (Add-ons -> Secrets) and this cell will use it automatically; otherwise it clones anonymously.
import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None

repo_url = f"https://{token}@github.com/pryndor/VigiLLM.git" if token else "https://github.com/pryndor/VigiLLM.git"
!rm -rf VigiLLM
!git clone --depth 1 $repo_url
%cd VigiLLM

In [ ]:
# Kaggle's base image already ships torch+CUDA; only add what's missing.
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# Generate the synthetic scenario seeds (data/synthetic_scenarios.jsonl is gitignored,
# so it isn't in the clone -- regenerate it here instead of shipping it).
# --seed 42 is the script's default, so this reproduces the same 500 scenarios
# used for the local pipeline test batch.
!python scripts/generate_scenarios.py --count 500 --out data/synthetic_scenarios.jsonl

In [ ]:
# Narratives already confirmed working (20/20 generated cleanly last run) --
# causality is the open problem: 19/19 parsed causality answers disagreed
# with the rule engine (0 agreement), which is statistically off given the
# rule engine itself picks "possible" for the majority of these scenarios.
# --skip-narrative to iterate on just the causality path without re-downloading
# BioMistral or re-spending GPU minutes on narratives each retry. New logging
# below prints model vs rule category side by side so the mismatch pattern
# is visible instead of just the ok/disagree verdict.
!python scripts/query_teachers.py --backend hf \
    --scenarios data/synthetic_scenarios.jsonl \
    --count 20 \
    --skip-narrative \
    --out data/distilled_training_set.jsonl \
    --flagged-out data/distilled_flagged.jsonl

In [ ]:
# Copy outputs to Kaggle's /kaggle/working so they show up in the notebook's Output tab
# and can be downloaded, then merged back into your local data/ dir.
!mkdir -p /kaggle/working/out
!cp data/distilled_training_set.jsonl data/distilled_flagged.jsonl /kaggle/working/out/
!wc -l /kaggle/working/out/*.jsonl